# SGZ A1q — L'habituation d'A1p passe-t-elle par les têtes d'induction ?

**Point de départ (A1p, commit 02a52e3).** Un jeton rare X forcé une seconde fois (t₂ = 96, après t₁ = 64) a un effet
direct plus faible qu'un jeton Y nouveau de même surprise : Sp = D_rep / D_nouv = 0,90 [0,86 ; 0,94] (Qwen3-1.7B-Base) et
0,88 [0,83 ; 0,93] (Qwen3-1.7B), sur k = 2–31. Analyse après coup : juste après X répété (k = 1, hors fenêtre), le
rapport tombe à 0,18 [0,10 ; 0,26] et 0,20 [0,12 ; 0,29]. **Lecture concurrente** : ce n'est pas une habituation mais la
recopie par les têtes d'induction (Olsson et al., 2022, arXiv:2209.11895) : déjà vu « X puis sa suite », le modèle
prédit la même suite.

**Méthode.** Mêmes prompts, même graine, mêmes trajectoires qu'A1p (ancrage MD5 sur les 768 × 128 jetons). Les passes
rejouées choc1, double et nouveau sont refaites sous trois conditions :
- **aucune** ablation (reproduction d'A1p) ;
- **induction** : sortie des K têtes au plus fort score de recopie mise à zéro (entrée de o_proj), à toutes les
  positions ; K = 3 % des têtes (comme Crosbie & Shutova, 2024, arXiv:2407.07011, qui ablatent 1 % et 3 %) ;
- **aléatoire** : 20 tirages de K autres têtes, appariés par couche (même nombre de têtes par couche que l'ensemble
  induction).

**Score de recopie (causal, sélection principale)** : sur 16 suites de 64 jetons aléatoires répétées deux fois, hausse
de la perte sur la seconde copie quand on retire cette tête seule, moins sa hausse sur la première copie (dégât
général). **Score d'attention (Olsson et al., descriptif)** : attention moyenne de la position i de la seconde copie vers
i − 64 + 1 ; ses K premières têtes forment une condition descriptive « motif », et le recouvrement des deux ensembles
est imprimé. Raison du choix, trouvée sur le harnais avant livraison : un Qwen3 minuscule entraîné à recopier avait
des scores d'attention de 0,01 à 0,05 partout, alors que retirer une seule de ses têtes faisait monter la perte de
recopie de 1,5 à 2,6 nats ; le score d'attention peut donc manquer les têtes qui recopient [établi sur ce jouet].
Antériorité de l'ablation (recherche du 28 oct.) : Crosbie & Shutova (2024) ; Rao (2026), sur GPT-2 small, trouve
qu'ablater les têtes d'induction change l'entropie de sortie de façon non additive, avec un seuil vers 3 à 5 têtes.
D'où une dose descriptive en plus : 1 % et 6 % des têtes.

## Critères (fixés avant toute exécution)

Pour chaque condition c : D_rep,c = double − choc1 et D_nouv,c = nouveau − choc1 (entropie de Tsallis q = 1,45, passes
rejouées), Sp_c = moyenne(D_rep,c) / moyenne(D_nouv,c) sur k = 2–31, R1_c = même rapport à k = 1. ΔSp = Sp_induction −
Sp_aucune et ΔR1 = R1_induction − R1_aucune, IC95 par bootstrap apparié sur les prompts (B = 1000) ; pour chaque tirage
aléatoire j, ΔSp_j et ΔR1_j (estimations ponctuelles). Mêmes prompts exclus qu'A1p (X tiré par le modèle à t₂).
- **M1, l'ablation casse-t-elle l'induction ?** Hausse de perte sur la seconde copie des suites répétées (suites
  neuves, distinctes de celles qui servent à choisir les têtes). EFFICACE si elle est positive et supérieure aux 20 tirages aléatoires ;
  sinon INEFFICACE, et V1, V2 deviennent NON TESTABLE.
- **V1, recopie à k = 1.** PORTÉE PAR L'INDUCTION si IC95(ΔR1) > 0 et ΔR1 supérieur aux 20 ΔR1_j ; NON SPÉCIFIQUE si
  IC95(ΔR1) > 0 sans dépasser les 20 tirages ; INVERSE si IC95(ΔR1) < 0 ; sinon INCHANGÉE.
- **V2, habituation sur k = 2–31.** NON TESTABLE si M1 est INEFFICACE ou si l'IC95 de D_nouv sous ablation contient 0.
  EXPLIQUÉE si IC95(ΔSp) > 0, ΔSp supérieur aux 20 ΔSp_j et la borne haute de l'IC95 de Sp_induction ≥ 1 (plus
  d'habituation détectable) ; RÉDUITE si IC95(ΔSp) > 0 et ΔSp supérieur aux 20 tirages, mais IC95(Sp_induction) < 1 ;
  INDÉPENDANTE si IC95(Sp_induction) < 1, ΔSp ne remplit pas la condition précédente et V1 = PORTÉE PAR L'INDUCTION
  (la recopie est cassée, l'habituation reste) ; sinon INDÉTERMINÉE.
- **Synthèse, thèse « l'habituation d'A1p passe par les têtes d'induction »** : SOUTENUE si V2 = EXPLIQUÉE pour les deux
  modèles ; PARTIELLE si V2 ∈ {EXPLIQUÉE, RÉDUITE} pour les deux sans être EXPLIQUÉE deux fois ; RÉFUTÉE si V2 =
  INDÉPENDANTE pour les deux ; DÉPEND DU MODÈLE si l'un est EXPLIQUÉE ou RÉDUITE et l'autre INDÉPENDANTE ; sinon
  INDÉTERMINÉE.
- **Descriptifs** : Sp et R1 aux doses 1 % et 6 % et pour l'ensemble « motif » (score d'attention) ; profil par tranches de k ; entropie de base de choc1 sous chaque
  ablation (dégât général) ; témoin − référence : distribution des ΔSp_j et ΔR1_j.
- Témoins positifs synthétiques pour chaque règle ; vérification du crochet (ablation par crochet = colonnes de o_proj
  mises à zéro ; masque plein = aucun écart) ; ancrage MD5 sur A1p.

## Prédictions de l'assistant (écrites avant exécution) [spéculation]

M1 EFFICACE : 85 % par modèle. V1 PORTÉE PAR L'INDUCTION : base 75 %, instruct 70 %. V2 : EXPLIQUÉE 20 %, RÉDUITE 35 %,
INDÉPENDANTE 25 %, INDÉTERMINÉE 20 %. Synthèse : SOUTENUE 15 %, PARTIELLE 30 %, RÉFUTÉE 20 %, DÉPEND DU MODÈLE 15 %.

## Limites connues

- Ablation par mise à zéro (pas par la moyenne) : elle sort le modèle de sa distribution ; le témoin aléatoire apparié
  par couche contrôle le dégât générique, pas un dégât propre aux têtes à fort score.
- Les deux scores sont mesurés sur des jetons aléatoires ; une tête peut recopier dans le texte sans être retenue.
- Les têtes sont ablatées à toutes les positions, donc aussi pendant la lecture du premier choc.
- Une seule famille (Qwen3), une taille, une température (0,6), un écart entre chocs (32 jetons).

## Réglages Kaggle

Accélérateur **GPU T4 x2**, **Internet activé** (aucun jeton Hugging Face requis). Durée estimée : ~45 à 55 min par
modèle [estimation]. **À lire en priorité** : `[TÉMOIN POSITIF]`, `[ANCRAGE]`, `[VÉRIF]`, `[INDUCTION]`, `[M1]`,
`[EFFETS]`, `[V1]`, `[V2]`, `[SYNTHÈSE]`.

In [ ]:
import gc
import hashlib
import json
import math
import os
import sys
import time

import numpy as np
import torch

try:
    sys.stdout.reconfigure(line_buffering=True)
except Exception:
    pass

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"
OUT = os.environ.get("OUT", "/kaggle/working/sgz_a1q" if os.path.isdir("/kaggle/working") else "sgz_a1q_out")
HF_CACHE = os.environ.get("HF_CACHE", "/tmp/hf_cache")
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()

T_MAIN = [0.6]
Q = 1.45
N_PROMPTS, GEN_LEN, T1, T2 = 768, 128, 64, 96
RANK_LO, RANK_HI = 1000, 5000
MID_D = (2, 31)
BOOT_N = int(os.environ.get("BOOT_N", "1000"))
SEED = 20261025                                   # même graine que A1c–A1p : mêmes trajectoires
MAX_HOURS = float(os.environ.get("MAX_HOURS", "3"))
K_FRAC, K_DOSES = 0.03, (0.01, 0.06)              # part des têtes ablatées : principale, puis doses descriptives
N_ALEA = 20                                       # tirages de têtes aléatoires appariées par couche
L_IND, B_IND = 64, 16                             # suites aléatoires répétées : longueur d'une copie, nombre de suites
MODELES_A_LANCER = ["Qwen/Qwen3-1.7B-Base", "Qwen/Qwen3-1.7B"]
MODELS = [(m.split("/")[-1], [m]) for m in MODELES_A_LANCER]
A1P_MD5 = {"Qwen3-1.7B-Base": {"temoin": "c70f088d2a2549962056c5b5b476abc0", "choc1": "5def8870f237088cc74b1a7288d0ebcb",
                               "X": "502ed8b1685a2d485720794a84d3bf32", "Y": "41602bff40d36d481006f15919bd539d"},
           "Qwen3-1.7B": {"temoin": "bfbb69b097d734abeee0eef74aadd94a", "choc1": "0d8e8d827040ce0ec4f9242a5408c621",
                          "X": "18c44342cdff0bfc83a590cdca3a95a3", "Y": "3c8cb9840987525b2b1deef91bc52756"}}
A1P_REF = {"Qwen3-1.7B-Base": {"Sp": 0.8995, "R1": 0.18}, "Qwen3-1.7B": {"Sp": 0.8752, "R1": 0.20}}
BRAS_R = ["choc1", "double", "nouveau"]
if os.environ.get("MODEL_IDS"):
    MODELS = [(m.split("/")[-1], [m]) for m in os.environ["MODEL_IDS"].split(";")]
if SMOKE_TEST:
    N_PROMPTS, GEN_LEN, T1, T2 = 16, 60, 16, 32
    RANK_LO, RANK_HI = 20, 80
    MID_D = (2, 12)
    BOOT_N = 50
    N_ALEA, L_IND, B_IND = 3, 12, 16
    K_FRAC, K_DOSES = 0.25, (0.125, 0.375)
    MODELS = [("smoke-qwen3-aleatoire", ["smoke-qwen3-random"]), ("smoke-qwen3-induction", ["smoke-qwen3-induction"])]
NPOS = MID_D[1] + 1                               # entropies rejouées gardées : indices t₂ … t₂ + 31
assert T2 + MID_D[1] <= GEN_LEN - 1, "fenêtre après t₂ trop longue pour la génération"
if not SMOKE_TEST and DEVICE != "cuda":
    raise RuntimeError("Aucun GPU détecté : active l'accélérateur « GPU T4 x2 » (Settings → Accelerator).")
torch.manual_seed(SEED)
np.random.seed(SEED)
import transformers
print(f"[CONFIG] SMOKE_TEST={SMOKE_TEST} DEVICE={DEVICE} N_GPU={N_GPU} OUT={OUT}")
print(f"[CONFIG] T={T_MAIN} prompts={N_PROMPTS} longueur={GEN_LEN} t₁={T1} t₂={T2} fenêtre k={MID_D[0]}–{MID_D[1]} ; "
      f"K = {K_FRAC:.0%} des têtes (doses {', '.join(f'{d:.0%}' for d in K_DOSES)}), {N_ALEA} tirages aléatoires ; "
      f"modèles={[m[0] for m in MODELS]}")
print(f"[CONFIG] torch {torch.__version__}, transformers {transformers.__version__}")

In [ ]:
# ---------- Prompts ----------
PROMPTS_MAIN = [
    "La ville de Lyon est située au confluent",
    "Le principe de conservation de l'énergie affirme que",
    "Hier soir, en rentrant du travail, Marie a découvert",
    "Pour préparer une pâte à crêpes, il faut d'abord",
    "L'histoire de la Révolution française commence",
    "Les réseaux de neurones artificiels sont des modèles",
    "Le chat s'approcha lentement de la fenêtre et",
    "Dans un article publié la semaine dernière, les chercheurs",
    "La photosynthèse est le processus par lequel",
    "Il était une fois, dans un petit village de montagne,",
    "Le théorème de Pythagore permet de calculer",
    "Selon les prévisions météorologiques, demain il",
    "Le marché boursier a connu une forte baisse",
    "Pour installer ce logiciel, ouvrez un terminal et",
    "La musique baroque se caractérise par",
    "Le fleuve Saint-Laurent traverse",
    "Mon grand-père me racontait souvent que",
    "Les volcans se forment lorsque",
    "La recette traditionnelle du pot-au-feu",
    "En informatique, une fonction récursive est",
    "Le Québec compte plusieurs régions, dont",
    "La lumière se propage dans le vide à",
    "Le roman commence par une scène où",
    "Les abeilles jouent un rôle essentiel dans",
    "Le conseil municipal a voté hier une résolution",
    "Une étoile à neutrons est un objet",
    "Le premier jour d'école, Lucas avait peur de",
    "Pour résoudre une équation du second degré,",
    "La cathédrale Notre-Dame de Paris a été",
    "Les marées sont causées principalement par",
    "Le joueur a marqué un but à la dernière minute",
    "La langue française comporte de nombreux mots",
    "The city of Boston is known for",
    "In physics, the second law of thermodynamics states that",
    "Yesterday afternoon, the committee decided to",
    "To bake a loaf of bread, you first need to",
    "The history of the Roman Empire begins with",
    "Machine learning models are trained by",
    "The dog ran across the field and",
    "According to a recent study published in Nature,",
    "Photosynthesis converts sunlight into",
    "Once upon a time, in a kingdom by the sea,",
    "The Pythagorean theorem relates the sides of",
    "The weather forecast for tomorrow predicts",
    "Stock markets around the world fell sharply after",
    "To install the package, run the following command",
    "Jazz music originated in the early twentieth century",
    "The Amazon River flows through",
    "My grandmother used to tell me stories about",
    "Earthquakes occur when tectonic plates",
    "A classic recipe for tomato soup",
    "In computer science, a hash table is",
    "Canada is divided into ten provinces, including",
    "The speed of sound in air depends on",
    "The novel opens with a scene in which",
    "Honeybees communicate with each other by",
    "The city council approved a new budget",
    "A black hole is a region of spacetime",
    "On her first day at the new job, Sarah",
    "To solve a system of linear equations,",
    "The Eiffel Tower was built for",
    "Ocean currents are driven mainly by",
    "The team won the championship after",
    "The English language borrowed many words from",
]
LANG_MAIN = ["fr"] * 32 + ["en"] * 32
THEMES_FR = ["la photosynthèse", "la révolution industrielle", "les volcans", "le jazz", "la cuisine italienne", "les trous noirs",
             "le hockey sur glace", "les abeilles", "l'intelligence artificielle", "le Saint-Laurent", "la Renaissance",
             "les marées", "le vélo en ville", "les vaccins", "la Grèce antique", "les forêts boréales", "le théâtre",
             "l'électricité", "les échecs", "le chocolat", "les ponts suspendus", "la mémoire humaine", "les comètes",
             "le recyclage"]
TECH_FR = ["la chromodynamique quantique", "la catalyse enzymatique", "les réseaux de neurones convolutifs",
           "la cryptographie à clé publique", "la thermodynamique hors équilibre", "la spectroscopie par résonance magnétique nucléaire",
           "la topologie algébrique", "la régulation épigénétique", "les supraconducteurs à haute température",
           "la compilation à la volée", "la turbulence en mécanique des fluides", "l'inférence bayésienne variationnelle",
           "la tectonique des plaques", "les ondes gravitationnelles", "la photolithographie des semi-conducteurs",
           "la théorie des catégories", "l'édition génomique par CRISPR", "les codes correcteurs d'erreurs",
           "la relativité générale", "l'apprentissage par renforcement profond"]
DEBUTS_FR = ["Voici ce qu'il faut savoir sur {t} :", "Hier, un reportage consacré à {t} expliquait que",
             "Pour bien comprendre {t}, il faut d'abord", "L'histoire de {t} commence",
             "Dans ce cours avancé sur {t}, nous montrons que", "Une revue de littérature récente sur {t} conclut que",
             "Le formalisme mathématique de {t} repose sur", "Une erreur fréquente à propos de {t} consiste à"]
THEMES_EN = ["photosynthesis", "the Roman Empire", "earthquakes", "blues music", "Japanese cooking", "neutron stars",
             "basketball", "ants", "machine learning", "the Mississippi River", "the Industrial Revolution", "ocean currents",
             "urban gardening", "antibiotics", "ancient Egypt", "rainforests", "opera", "magnetism", "poker", "coffee",
             "skyscrapers", "human sleep", "asteroids", "solar power"]
TECH_EN = ["quantum chromodynamics", "enzyme catalysis", "convolutional neural networks", "public-key cryptography",
           "non-equilibrium thermodynamics", "nuclear magnetic resonance spectroscopy", "algebraic topology",
           "epigenetic regulation", "high-temperature superconductors", "just-in-time compilation", "turbulent fluid dynamics",
           "variational Bayesian inference", "plate tectonics", "gravitational waves", "semiconductor photolithography",
           "category theory", "CRISPR genome editing", "error-correcting codes", "general relativity",
           "deep reinforcement learning"]
DEBUTS_EN = ["Here is what you need to know about {t}:", "Yesterday, a documentary about {t} explained that",
             "To really understand {t}, you first need to", "The history of {t} begins",
             "In this advanced lecture on {t}, we show that", "A recent literature review on {t} concludes that",
             "The mathematical formalism of {t} relies on", "A common misconception about {t} is that"]


def fr_contract(s):
    """Contractions françaises : « de le » → « du », « à les » → « aux »…"""
    for a, b in ((" de le ", " du "), (" de les ", " des "), (" à le ", " au "), (" à les ", " aux ")):
        s = s.replace(a, b)
    return s


def templates(themes, tech, debuts, lang):
    out = []
    for i, t in enumerate(themes + tech):
        for d in debuts:
            s = d.format(t=t)
            out.append((fr_contract(s) if lang == "fr" else s, lang, i >= len(themes), i - len(themes)))
    return out


TPL_FR, TPL_EN = templates(THEMES_FR, TECH_FR, DEBUTS_FR, "fr"), templates(THEMES_EN, TECH_EN, DEBUTS_EN, "en")
TPL = [x for pair in zip(TPL_FR, TPL_EN) for x in pair]      # français et anglais alternés
ALL_P = [(p, l, False, -1) for p, l in zip(PROMPTS_MAIN, LANG_MAIN)] + TPL
assert len({p for p, *_ in ALL_P}) == len(ALL_P), "prompts en double"
assert not any(x in p for p, *_ in ALL_P for x in (" de le ", " de les ", " à le ", " à les "))
ALL_P = ALL_P[:N_PROMPTS]
PROMPTS = [p for p, *_ in ALL_P]
LANG = [l for _, l, *_ in ALL_P]
print(f"[PROMPTS] {len(PROMPTS)} prompts ({min(64, len(PROMPTS))} écrits à la main, {max(0, len(PROMPTS) - 64)} gabarits), mêmes que A1c–A1k")

In [ ]:
# ---------- Mesures, règles de décision et témoins positifs ----------
@torch.no_grad()
def entropies_from_logits(logits, qs):
    """Entropies du softmax complet (float64). q = 1 : Shannon (nats) ; sinon Tsallis S_q = (1 − Σ p^q)/(q − 1)."""
    logp = torch.log_softmax(logits.double(), dim=-1)
    p = logp.exp()
    out = {}
    for q in qs:
        if q == 1.0:
            out[q] = -(p * torch.where(p > 0, logp, torch.zeros_like(logp))).sum(-1)
        else:
            out[q] = (1.0 - (p ** q).sum(-1)) / (q - 1.0)
    return out


def gumbel_noise(V, seeds, device):
    """Bruit de Gumbel par ligne, graine explicite par ligne (mêmes nombres pour tous les bras d'un prompt)."""
    g = torch.Generator(device=device)
    rows = []
    for s in seeds:
        g.manual_seed(int(s))
        rows.append(torch.rand(V, generator=g, device=device, dtype=torch.float32))
    u = torch.stack(rows).clamp_(1e-10, 1 - 1e-7)
    return -torch.log(-torch.log(u))


def ci(v):
    v = np.asarray(v, dtype=float)
    v = v[np.isfinite(v)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if v.size else (float("nan"), float("nan"))


def win(X, w):
    return X[:, w[0] - 1:w[1]].mean(1)


def md5(a):
    return hashlib.md5(np.ascontiguousarray(np.asarray(a).astype(np.int64)).tobytes()).hexdigest()




def rapport_ci(x, y, boots):
    """r = moyenne(x) / moyenne(y) et son IC95 (bootstrap apparié sur les prompts)."""
    return float(x.mean() / y.mean()), ci([x[ix].mean() / y[ix].mean() for ix in boots])


def ecart_ci(xa, ya, xb, yb, boots):
    """(moy xa / moy ya) − (moy xb / moy yb), mêmes rééchantillons pour les deux conditions."""
    d = float(xa.mean() / ya.mean() - xb.mean() / yb.mean())
    return d, ci([xa[ix].mean() / ya[ix].mean() - xb[ix].mean() / yb[ix].mean() for ix in boots])


def regle_m1(d_ind, d_alea):
    return "EFFICACE" if d_ind > 0 and d_ind > max(d_alea) else "INEFFICACE"


def regle_v1(d, dci, d_alea, m1, interpretable=True):
    if not interpretable:
        return "NON INTERPRÉTABLE"
    if m1 != "EFFICACE":
        return "NON TESTABLE (ablation inefficace)"
    if dci[0] > 0:
        return "PORTÉE PAR L'INDUCTION" if d > max(d_alea) else "NON SPÉCIFIQUE"
    if dci[1] < 0:
        return "INVERSE"
    return "INCHANGÉE"


def regle_v2(sp_ci, d, dci, d_alea, nouv_ci, v1, m1, interpretable=True):
    if not interpretable:
        return "NON INTERPRÉTABLE"
    if m1 != "EFFICACE":
        return "NON TESTABLE (ablation inefficace)"
    if nouv_ci[0] <= 0 <= nouv_ci[1]:
        return "NON TESTABLE (D_nouv non significatif sous ablation)"
    fort = dci[0] > 0 and d > max(d_alea)
    if fort:
        return "EXPLIQUÉE" if sp_ci[1] >= 1 else "RÉDUITE"
    if sp_ci[1] < 1 and v1 == "PORTÉE PAR L'INDUCTION":
        return "INDÉPENDANTE"
    return "INDÉTERMINÉE"


def synthese(v_par_modele, interpretable=True):
    vs = list(v_par_modele.values())
    if not interpretable:
        return "NON INTERPRÉTABLE"
    if not vs:
        return "INDÉTERMINÉE"
    oui = ("EXPLIQUÉE", "RÉDUITE")
    if all(v == "EXPLIQUÉE" for v in vs):
        return "SOUTENUE"
    if all(v in oui for v in vs):
        return "PARTIELLE"
    if all(v == "INDÉPENDANTE" for v in vs):
        return "RÉFUTÉE"
    if any(v in oui for v in vs) and any(v == "INDÉPENDANTE" for v in vs):
        return "DÉPEND DU MODÈLE"
    return "INDÉTERMINÉE"


def effets_cond(SR):
    """D_rep, D_nouv par prompt : moyenne sur k = 2–31 (colonne k = indice t₂ + k) et valeur à k = 1."""
    rep, nouv = SR["double"] - SR["choc1"], SR["nouveau"] - SR["choc1"]
    w = slice(MID_D[0], MID_D[1] + 1)
    return {"rep": rep[:, w].mean(1), "nouv": nouv[:, w].mean(1), "rep1": rep[:, 1], "nouv1": nouv[:, 1],
            "S0": SR["choc1"][:, w].mean(1)}


def juger(E, boots, m1, interpretable=True):
    """E : dict condition → effets_cond (prompts retenus). Verdicts V1 et V2 et leurs chiffres."""
    a, i = E["aucune"], E["induction"]
    al = [E[c] for c in E if c.startswith("alea_")]
    sp0, sp0_ci = rapport_ci(a["rep"], a["nouv"], boots)
    spi, spi_ci = rapport_ci(i["rep"], i["nouv"], boots)
    r10, r10_ci = rapport_ci(a["rep1"], a["nouv1"], boots)
    r1i, r1i_ci = rapport_ci(i["rep1"], i["nouv1"], boots)
    dsp, dsp_ci = ecart_ci(i["rep"], i["nouv"], a["rep"], a["nouv"], boots)
    dr1, dr1_ci = ecart_ci(i["rep1"], i["nouv1"], a["rep1"], a["nouv1"], boots)
    dsp_al = [float(e["rep"].mean() / e["nouv"].mean() - sp0) for e in al]
    dr1_al = [float(e["rep1"].mean() / e["nouv1"].mean() - r10) for e in al]
    nouv_ci = ci([i["nouv"][ix].mean() for ix in boots])
    v1 = regle_v1(dr1, dr1_ci, dr1_al, m1, interpretable)
    v2 = regle_v2(spi_ci, dsp, dsp_ci, dsp_al, nouv_ci, v1, m1, interpretable)
    return {"Sp_aucune": (sp0, sp0_ci), "Sp_induction": (spi, spi_ci), "R1_aucune": (r10, r10_ci),
            "R1_induction": (r1i, r1i_ci), "dSp": (dsp, dsp_ci), "dR1": (dr1, dr1_ci), "dSp_alea": dsp_al,
            "dR1_alea": dr1_al, "D_nouv_induction_ci": nouv_ci, "V1": v1, "V2": v2}


# Témoins positifs : effets synthétiques par prompt, structure appariée entre conditions (même sensibilité par prompt)
rng_pc = np.random.default_rng(SEED + 1)
Pp = 768
bt = [rng_pc.integers(0, Pp, Pp) for _ in range(200)]
sens = rng_pc.lognormal(np.log(0.05), 0.5, Pp)


def synth(sp, r1, n_al=20, sp_al=0.88, r1_al=0.19, sp0=0.88, r10=0.19):
    """Sans ablation et ablation induction partagent leur bruit (mêmes prompts) ; chaque tirage aléatoire a le sien."""
    def bruits():
        return [rng_pc.lognormal(0, 0.1, Pp) for _ in range(4)]

    def cond(s, r, b):
        nouv, nouv1 = sens * b[0], 12 * sens * b[1]
        return {"rep": s * nouv * b[2], "nouv": nouv, "rep1": r * nouv1 * b[3], "nouv1": nouv1, "S0": sens}
    commun = bruits()
    E = {"aucune": cond(sp0, r10, commun), "induction": cond(sp, r1, commun)}
    for j in range(n_al):
        E[f"alea_{j}"] = cond(sp_al, r1_al, bruits())
    return E


checks = []
j = juger(synth(1.0, 0.8), bt, "EFFICACE")
checks += [("recopie portée", j["V1"], "PORTÉE PAR L'INDUCTION"), ("habituation expliquée", j["V2"], "EXPLIQUÉE")]
checks.append(("habituation réduite", juger(synth(0.95, 0.8), bt, "EFFICACE")["V2"], "RÉDUITE"))
checks.append(("habituation indépendante", juger(synth(0.88, 0.8), bt, "EFFICACE")["V2"], "INDÉPENDANTE"))
j = juger(synth(0.88, 0.19), bt, "EFFICACE")
checks += [("recopie inchangée", j["V1"], "INCHANGÉE"), ("sans recopie cassée", j["V2"], "INDÉTERMINÉE")]
checks.append(("recopie non spécifique", juger(synth(0.88, 0.8, r1_al=0.8), bt, "EFFICACE")["V1"], "NON SPÉCIFIQUE"))
checks.append(("habituation non spécifique", juger(synth(1.0, 0.8, sp_al=1.0), bt, "EFFICACE")["V2"], "INDÉTERMINÉE"))
checks.append(("ablation inefficace", juger(synth(1.0, 0.8), bt, "INEFFICACE")["V2"], "NON TESTABLE (ablation inefficace)"))
checks += [("M1 efficace", regle_m1(0.9, [0.1, 0.2]), "EFFICACE"), ("M1 inefficace", regle_m1(0.15, [0.1, 0.2]), "INEFFICACE"),
           ("synthèse soutenue", synthese({"a": "EXPLIQUÉE", "b": "EXPLIQUÉE"}), "SOUTENUE"),
           ("synthèse partielle", synthese({"a": "EXPLIQUÉE", "b": "RÉDUITE"}), "PARTIELLE"),
           ("synthèse réfutée", synthese({"a": "INDÉPENDANTE", "b": "INDÉPENDANTE"}), "RÉFUTÉE"),
           ("synthèse dépend", synthese({"a": "RÉDUITE", "b": "INDÉPENDANTE"}), "DÉPEND DU MODÈLE")]
for nom, got, att in checks:
    print(f"[TÉMOIN POSITIF] {nom:<26} : {got:<24} (attendu {att})")
POSITIVE_CONTROL_OK = all(g == e for _, g, e in checks)
print(f"[TÉMOIN POSITIF] {'RÉUSSI' if POSITIVE_CONTROL_OK else 'RATÉ : verdicts en NON TESTABLE'}")

In [ ]:
# ---------- Chargement ----------
class SmokeTokenizer:
    """Tokenizer jouet par caractères, aller-retour exact (SMOKE_TEST)."""
    ALPHA = ("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,;:!?'\"()-«»\n"
             "àâäçéèêëîïôöùûüÀÇÉÈÊ")

    def __init__(self, V):
        self.V, self.pad_token_id, self.eos_token_id = V, 0, 1
        self.all_special_ids = [0, 1]

    def __len__(self):                          # jetons valides : l'alphabet (le reste du vocabulaire est masqué)
        return 2 + len(self.ALPHA)

    def encode(self, s, **kw):
        return [2 + self.ALPHA.index(c) if c in self.ALPHA else 2 + self.ALPHA.index("?") for c in s]

    def decode(self, ids, **kw):
        return "".join(self.ALPHA[i - 2] if 2 <= i < 2 + len(self.ALPHA) else "?" for i in ids)


def transformers_major():
    return int(transformers.__version__.split(".")[0])


def load_model(model_id, fp32=False):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if model_id == "smoke-qwen3-random":
        from transformers import Qwen3Config, Qwen3ForCausalLM
        cfg = Qwen3Config(vocab_size=512, hidden_size=64, intermediate_size=128, num_hidden_layers=4,
                          num_attention_heads=4, num_key_value_heads=2, head_dim=16, max_position_embeddings=512)
        torch.manual_seed(SEED)
        return Qwen3ForCausalLM(cfg).eval().to(DEVICE), SmokeTokenizer(512), "fp32 aléatoire (smoke)"
    if model_id == "smoke-qwen3-induction":
        # Qwen3 minuscule entraîné à recopier des suites aléatoires répétées : il forme de vraies têtes d'induction
        from transformers import Qwen3Config, Qwen3ForCausalLM
        cfg = Qwen3Config(vocab_size=512, hidden_size=128, intermediate_size=256, num_hidden_layers=2,
                          num_attention_heads=4, num_key_value_heads=2, head_dim=32, max_position_embeddings=512)
        torch.manual_seed(SEED)
        m, tk = Qwen3ForCausalLM(cfg).to(DEVICE), SmokeTokenizer(512)
        opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
        rng = np.random.default_rng(SEED + 99)
        for _ in range(int(os.environ.get("SMOKE_PAS", "4000"))):
            r = rng.integers(2, len(tk), (64, int(rng.integers(6, 25))))   # longueur variable : pas de raccourci positionnel
            x = torch.tensor(np.concatenate([r, r], 1), device=DEVICE)
            loss = m(input_ids=x, labels=x).loss
            opt.zero_grad()
            loss.backward()
            opt.step()
        return m.eval(), tk, f"fp32 entraîné à recopier (smoke, perte finale {loss.item():.2f})"
    tok = AutoTokenizer.from_pretrained(model_id, cache_dir=HF_CACHE)
    dkey = "dtype" if transformers_major() >= 5 else "torch_dtype"
    kw = {"cache_dir": HF_CACHE}
    if DEVICE == "cpu":
        kw[dkey], mode = torch.float32, "fp32 CPU"
    elif fp32:
        kw[dkey], kw["device_map"] = torch.float32, ("auto" if N_GPU >= 2 else {"": 0})
        kw["max_memory"] = {i: ("12GiB" if i == 0 else "13GiB") for i in range(N_GPU)}
        mode = f"fp32 sur {N_GPU} GPU"
    elif any(s in model_id for s in ("7B", "8B", "9B", "14B")) and N_GPU >= 2:
        kw[dkey], kw["device_map"] = torch.float16, "auto"
        kw["max_memory"] = {i: ("12GiB" if i == 0 else "13GiB") for i in range(N_GPU)}
        mode = f"fp16 sur {N_GPU} GPU"
    else:
        kw[dkey], kw["device_map"], mode = torch.float16, {"": 0}, "fp16 1 GPU"
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()
    if DEVICE == "cuda":
        assert next(model.parameters()).dtype == (torch.float32 if fp32 else torch.float16), "type de poids inattendu"
    return model, tok, mode


def set_attention(model, impl):
    """Génération avec l'attention par défaut (comme A1c), passes rejouées en « eager » (poids d'attention visibles)."""
    try:
        model.set_attn_implementation(impl)
    except Exception:
        model.config._attn_implementation = impl
    return getattr(model.config, "_attn_implementation", impl)

In [ ]:
# ---------- Génération (identique à A1c–A1k) ----------
class Gen:
    def __init__(self, model, tok, label):
        self.model, self.tok, self.label = model, tok, label
        self.dev = model.get_input_embeddings().weight.device
        self.V = model.get_output_embeddings().weight.shape[0]
        forbid = set(tok.all_special_ids or []) | {tok.eos_token_id}
        ge = getattr(getattr(model, "generation_config", None), "eos_token_id", None)
        forbid |= set(ge if isinstance(ge, (list, tuple)) else ([ge] if ge is not None else []))
        forbid.discard(None)
        self.forbid, self.n_tok = sorted(forbid), len(tok)
        self.pad = tok.pad_token_id if tok.pad_token_id is not None else 0
        npar = sum(p.numel() for p in model.parameters())
        self.budget = 48 if npar < 3e9 else 32
        self.t_first = None

    def left_pad(self, seqs):
        L = max(len(s) for s in seqs)
        ids = torch.full((len(seqs), L), self.pad, dtype=torch.long)
        attn = torch.zeros((len(seqs), L), dtype=torch.long)
        for j, s in enumerate(seqs):
            ids[j, L - len(s):] = torch.tensor(s, dtype=torch.long)
            attn[j, L - len(s):] = 1
        return ids.to(self.dev), attn.to(self.dev), L

    @torch.no_grad()
    def sample(self, ctxs, prompt_of, T, nsteps, step0, force=None):
        n = len(ctxs)
        S, tk = np.zeros((n, nsteps), np.float32), np.zeros((n, nsteps), np.int64)
        ids, attn, _ = self.left_pad(ctxs)
        pos = (attn.cumsum(-1) - 1).clamp(min=0)
        past = None
        for s in range(nsteps):
            out = self.model(input_ids=ids, attention_mask=attn, position_ids=pos, past_key_values=past, use_cache=True)
            past = out.past_key_values
            z = out.logits[:, -1, :].float()
            if not torch.isfinite(z).all():
                raise FloatingPointError(f"logits non finis à T={T}, pas {s}")
            e1 = entropies_from_logits(z, [Q])[Q].cpu().numpy()
            seeds = [SEED + 100003 * prompt_of[j] + step0 + s for j in range(n)]
            score = z / T + gumbel_noise(self.V, seeds, z.device)
            score[:, self.forbid] = -float("inf")
            if self.V > self.n_tok:
                score[:, self.n_tok:] = -float("inf")
            nxt = torch.argmax(score, dim=-1)
            if force is not None:
                nxt = force(s, z, nxt)
            S[:, s], tk[:, s] = e1, nxt.cpu().numpy()
            ids = nxt[:, None].to(self.dev)
            attn = torch.cat([attn, torch.ones((n, 1), dtype=attn.dtype, device=self.dev)], dim=1)
            pos = pos[:, -1:] + 1
        del past, out
        return S, tk

    def progress(self, t_lot, n_lots_total):
        if self.t_first is None:
            self.t_first = t_lot
            est = t_lot * n_lots_total / 60
            print(f"[PROGRESSION] {self.label} : premier lot en {t_lot:.1f} s ; estimation ~{est:.0f} min (génération)", flush=True)
            if est / 60 > MAX_HOURS:
                raise TimeoutError(f"estimation {est:.0f} min > budget {MAX_HOURS} h")

In [ ]:
# ---------- Deux chocs : génération et passes rejouées ----------
def Gen_masque(self, score):
    """Mêmes jetons interdits que l'échantillonnage (spéciaux, fin de texte, hors vocabulaire)."""
    score[:, self.forbid] = -float("inf")
    if self.V > self.n_tok:
        score[:, self.n_tok:] = -float("inf")
    return score


Gen.masque = Gen_masque


@torch.no_grad()
def run_main(g, enc):
    """Témoin et choc1 empilés (comme A1c–A1k : X au rang tiré à t₁, jamais le jeton du témoin). À t₂, rien n'est
    forcé pendant la génération : on note la surprise de X dans les deux trajectoires et on choisit Y, au rang voisin
    de celui de X dans la distribution de choc1."""
    P = len(enc)
    forced_rank = np.random.default_rng(SEED + 5).integers(RANK_LO, RANK_HI + 1, P)
    T = T_MAIN[0]
    res = {a: {"S": np.zeros((P, GEN_LEN), np.float32), "tok": np.zeros((P, GEN_LEN), np.int64)} for a in ("temoin", "choc1")}
    info = {k: np.zeros(P, np.int64) for k in ("X", "Y", "rang1", "rangX2", "rangY2")}
    info.update({k: np.zeros(P, np.float64) for k in ("surpX1", "surpX2_temoin", "surpX2_choc1", "surpY2_choc1")})
    npb = max(1, g.budget // 2)
    n_lots = math.ceil(P / npb)
    t0 = time.time()
    for b0 in range(0, P, npb):
        tl = time.time()
        rows_p = list(range(b0, min(P, b0 + npb)))
        n = len(rows_p)

        def force(s, z, nxt, n=n, rows_p=rows_p):
            if s < T1:
                nxt[n:] = nxt[:n]                                      # appariement exact avant le premier choc
            elif s == T1:
                zz = g.masque(z[n:].clone())
                lp = torch.log_softmax(z[n:].double(), dim=-1)
                order = torch.argsort(zz, dim=-1, descending=True)
                for j, r in enumerate(rows_p):
                    rk = int(min(forced_rank[r], order.shape[1] - 2))
                    if int(order[j, rk]) == int(nxt[j]):             # jamais le jeton du témoin
                        rk += 1
                    x = int(order[j, rk])
                    nxt[n + j] = x
                    info["X"][r], info["rang1"][r], info["surpX1"][r] = x, rk, -float(lp[j, x])
            elif s == T2:
                lp = torch.log_softmax(z.double(), dim=-1)
                zz = g.masque(z[n:].clone())
                order = torch.argsort(zz, dim=-1, descending=True)
                for j, r in enumerate(rows_p):
                    x = int(info["X"][r])
                    rx = int((order[j] == x).nonzero()[0, 0])
                    ry = rx + 1 if rx + 1 < order.shape[1] else rx - 1
                    if int(order[j, ry]) == int(nxt[n + j]):         # Y n'est jamais le jeton tiré par choc1
                        ry = ry + 1 if ry + 1 < order.shape[1] else rx - 2
                    y = int(order[j, ry])
                    info["Y"][r], info["rangX2"][r], info["rangY2"][r] = y, rx, ry
                    info["surpX2_temoin"][r] = -float(lp[j, x])
                    info["surpX2_choc1"][r], info["surpY2_choc1"][r] = -float(lp[n + j, x]), -float(lp[n + j, y])
            return nxt

        S, tk = g.sample([enc[r] for r in rows_p] * 2, rows_p * 2, T, GEN_LEN, 0, force)
        for ai, a in enumerate(("temoin", "choc1")):
            sl = slice(ai * n, (ai + 1) * n)
            res[a]["S"][rows_p], res[a]["tok"][rows_p] = S[sl], tk[sl]
        g.progress(time.time() - tl, n_lots)
    ref, fo = res["temoin"]["tok"], res["choc1"]["tok"]
    same = np.array_equal(ref[:, :T1], fo[:, :T1])
    chg = float(np.mean(ref[:, T1] != fo[:, T1]))
    print(f"[VÉRIF] {g.label} : identique avant t₁ : {same} ; jeton changé à t₁ : {chg:.0%} ; rang médian du choc "
          f"{int(np.median(info['rang1']))} ; à t₂, rang médian de X après le premier choc {int(np.median(info['rangX2']))}",
          flush=True)
    if not same or chg < 0.99:
        raise RuntimeError("appariement ou choc défectueux : résultats non interprétables")
    print(f"[MESURE] {g.label} génération : {time.time() - t0:.0f} s", flush=True)
    return res, info

In [ ]:
# ---------- Ablation de têtes, score d'induction, passes rejouées ----------
class Ablation:
    """Met à zéro la sortie de têtes choisies (entrée de o_proj, avant la projection), à toutes les positions."""

    def __init__(self, model):
        self.layers = model.model.layers
        cfg = model.config
        self.nl, self.nh = len(self.layers), cfg.num_attention_heads
        self.hd = getattr(cfg, "head_dim", None) or cfg.hidden_size // self.nh
        assert self.layers[0].self_attn.o_proj.in_features == self.nh * self.hd, "o_proj : largeur d'entrée inattendue"
        self.masks = {}
        self.handles = [l.self_attn.o_proj.register_forward_pre_hook(self._hook(i)) for i, l in enumerate(self.layers)]

    def _hook(self, i):
        def f(mod, args):
            m = self.masks.get(i)
            if m is None:
                return None
            return (args[0] * m.to(dtype=args[0].dtype),) + tuple(args[1:])
        return f

    def set(self, heads, plein=None):
        self.masks = {}
        for l, h in heads:
            if l not in self.masks:
                self.masks[l] = torch.ones(self.nh * self.hd)
            self.masks[l][h * self.hd:(h + 1) * self.hd] = 0
        for l in plein or []:                                    # masque plein (aucune tête coupée) : vérification
            self.masks.setdefault(l, torch.ones(self.nh * self.hd))
        for l in self.masks:
            self.masks[l] = self.masks[l].to(self.layers[l].self_attn.o_proj.weight.device)

    def clear(self):
        self.masks = {}

    def remove(self):
        for h in self.handles:
            h.remove()


def suites_repetees(g, seed):
    """B_IND suites de L_IND jetons aléatoires, chacune répétée deux fois (jetons ordinaires, jamais interdits)."""
    rng = np.random.default_rng(seed)
    lo, hi = (2, g.n_tok) if SMOKE_TEST else (1000, min(g.n_tok, 100000))
    interdits = set(g.forbid)
    r = rng.integers(lo, hi, (B_IND, L_IND))
    for (b, t) in zip(*np.nonzero(np.isin(r, list(interdits)))):
        while r[b, t] in interdits:
            r[b, t] = rng.integers(lo, hi)
    return torch.tensor(np.concatenate([r, r], 1), dtype=torch.long, device=g.dev)


@torch.no_grad()
def score_induction(g, ids):
    """Score d'Olsson et al. : attention de la position i (seconde copie) vers i − L + 1, moyennée (couches × têtes)."""
    impl0 = getattr(g.model.config, "_attn_implementation", None) or "sdpa"
    eff = set_attention(g.model, "eager")
    try:
        att = g.model(input_ids=ids, output_attentions=True, use_cache=False).attentions
    finally:
        set_attention(g.model, impl0)
    assert att is not None and len(att) == len(g.model.model.layers) and att[0] is not None, \
        f"poids d'attention indisponibles (implémentation {eff})"
    L = ids.shape[1] // 2
    qi = torch.arange(L, 2 * L)
    return np.stack([a[:, :, qi.to(a.device), (qi - L + 1).to(a.device)].float().mean((0, 2)).cpu().numpy() for a in att])


@torch.no_grad()
def perte_copie(g, ids):
    """Perte moyenne (nats) sur la première copie et sur la seconde (hors son premier jeton) des suites répétées."""
    base, head = g.model.model, g.model.get_output_embeddings()
    h = base(input_ids=ids, use_cache=False).last_hidden_state
    L, nll = ids.shape[1] // 2, []
    for b in range(ids.shape[0]):
        lp = torch.log_softmax(head(h[b, :-1].to(head.weight.device)).float(), dim=-1)
        nll.append(-lp.gather(-1, ids[b, 1:, None].to(lp.device))[:, 0].cpu().numpy())
    nll = np.stack(nll)                                           # colonne t : prédiction du jeton t + 1
    return float(nll[:, :L - 1].mean()), float(nll[:, L:].mean())


@torch.no_grad()
def score_recopie(g, abl, ids):
    """Score causal par tête : hausse de perte sur la seconde copie quand on retire la tête seule, moins celle de la première."""
    abl.clear()
    f0, s0 = perte_copie(g, ids)
    sc = np.zeros((abl.nl, abl.nh))
    for l in range(abl.nl):
        for h in range(abl.nh):
            abl.set([(l, h)])
            f, s_ = perte_copie(g, ids)
            sc[l, h] = (s_ - s0) - (f - f0)
    abl.clear()
    return sc


def choisir(sc, k):
    return [tuple(int(v) for v in np.unravel_index(i, sc.shape)) for i in np.argsort(-sc, axis=None)[:k]]


def tirage_apparie(sel, nl, nh, rng):
    """Autant de têtes par couche que dans sel, tirées hors de sel ; couches voisines si une couche n'en a pas assez."""
    excl, out = set(sel), []
    compte = {}
    for l, _ in sel:
        compte[l] = compte.get(l, 0) + 1
    for l in sorted(compte):
        reste, d = compte[l], 0
        while reste > 0 and d < nl:
            for l2 in ((l,) if d == 0 else (l - d, l + d)):
                if 0 <= l2 < nl and reste > 0:
                    dispo = [(l2, h) for h in range(nh) if (l2, h) not in excl and (l2, h) not in out]
                    if dispo:
                        pris = rng.choice(len(dispo), min(reste, len(dispo)), replace=False)
                        out += [dispo[int(i)] for i in pris]
                        reste -= len(pris)
            d += 1
    return out


@torch.no_grad()
def rejouer(g, ctxs, conts, p0, npos):
    """Entropie aux indices p0 … p0 + npos − 1 de la suite imposée (indice i : loi du jeton i), une passe sans cache."""
    base, head = g.model.model, g.model.get_output_embeddings()
    n, Kp = len(ctxs), len(conts[0]) + 1
    ids, attn, L = g.left_pad([c + s for c, s in zip(ctxs, conts)])
    pos = (attn.cumsum(-1) - 1).clamp(min=0)
    h = base(input_ids=ids, attention_mask=attn, position_ids=pos, use_cache=False).last_hidden_state
    hk = h[:, L - Kp + p0:L - Kp + p0 + npos, :].reshape(n * npos, -1)
    flat = np.zeros(n * npos, np.float64)
    for c0 in range(0, n * npos, 192):
        flat[c0:c0 + 192] = entropies_from_logits(head(hk[c0:c0 + 192].to(head.weight.device)).float(), [Q])[Q].cpu().numpy()
    return flat.reshape(n, npos).astype(np.float32)


@torch.no_grad()
def run_replay(g, enc, res, info):
    """Bras choc1, double (X de nouveau à t₂) et nouveau (Y à t₂) rejoués sous l'ablation en cours ; indices t₂ … t₂ + 31."""
    P = len(enc)
    SR = {a: np.zeros((P, NPOS), np.float32) for a in BRAS_R}
    nb, lon = 12, T2 + NPOS - 1
    for b0 in range(0, P, nb):
        rows = list(range(b0, min(P, b0 + nb)))
        cc = [res["choc1"]["tok"][r, :lon].tolist() for r in rows]
        sub = lambda c, x: c[:T2] + [int(x)] + c[T2 + 1:]
        conts = {"choc1": cc, "double": [sub(c, info["X"][r]) for c, r in zip(cc, rows)],
                 "nouveau": [sub(c, info["Y"][r]) for c, r in zip(cc, rows)]}
        out = rejouer(g, [enc[r] for r in rows] * len(BRAS_R), [x for a in BRAS_R for x in conts[a]], T2, NPOS)
        m = len(rows)
        for ai, a in enumerate(BRAS_R):
            SR[a][rows] = out[ai * m:(ai + 1) * m]
    return SR


@torch.no_grad()
def verif_crochet(g, abl, res, enc, tete):
    """Ablation par le crochet = colonnes de o_proj mises à zéro ; masque plein = aucun écart ; ablation active."""
    rows = list(range(min(4, len(enc))))
    ids, attn, _ = g.left_pad([enc[r] + res["choc1"]["tok"][r, :T2].tolist() for r in rows])
    pos = (attn.cumsum(-1) - 1).clamp(min=0)
    run = lambda: g.model(input_ids=ids, attention_mask=attn, position_ids=pos, use_cache=False).logits[:, -1].float()
    abl.clear()
    z0 = run()
    abl.set([], plein=list(range(abl.nl)))
    z_plein = run()
    l, h = tete
    abl.set([tete])
    z_cro = run()
    abl.clear()
    W = g.model.model.layers[l].self_attn.o_proj.weight
    sauve = W[:, h * abl.hd:(h + 1) * abl.hd].clone()
    W[:, h * abl.hd:(h + 1) * abl.hd] = 0
    try:
        z_poids = run()
    finally:
        W[:, h * abl.hd:(h + 1) * abl.hd] = sauve
    z_fin = run()
    e_plein = float((z_plein - z0).abs().max())
    e_poids = float((z_cro - z_poids).abs().max())
    e_actif = float((z_cro - z0).abs().max())
    e_rest = float((z_fin - z0).abs().max())
    print(f"[VÉRIF] {g.label} crochet : masque plein contre aucun {e_plein:.1e} ; crochet contre poids mis à zéro "
          f"{e_poids:.1e} ; effet de l'ablation d'une tête {e_actif:.1e} ; après restauration {e_rest:.1e}", flush=True)
    if e_plein > 1e-3 or e_rest > 1e-3 or e_poids > max(1e-3, 0.05 * e_actif) or not e_actif > 1e-3:
        raise RuntimeError("crochet d'ablation défectueux : résultats non interprétables")

In [ ]:
# ---------- Boucle principale ----------
interpretable = not SMOKE_TEST
ok = interpretable and POSITIVE_CONTROL_OK
ALL = {}


def ecrire_resume():
    with open(os.path.join(OUT, "resume_a1q.json"), "w") as fh:
        json.dump({"smoke": SMOKE_TEST, "positive_control_ok": POSITIVE_CONTROL_OK, **ALL}, fh,
                  ensure_ascii=False, indent=1, default=str)


def nom_tete(t):
    return f"L{t[0]}.H{t[1]}"


def tranches(SR, valide):
    rep, nouv = (SR["double"] - SR["choc1"])[valide], (SR["nouveau"] - SR["choc1"])[valide]
    return [float(rep[:, lo:min(hi, MID_D[1]) + 1].mean() / nouv[:, lo:min(hi, MID_D[1]) + 1].mean())
            for lo, hi in ((1, 1), (2, 7), (8, 15), (16, 31)) if lo <= MID_D[1]]


for label, ids in MODELS:
    t_start = time.time()
    model = tok = g = abl = None
    try:
        for mid in ids:
            print(f"\n[CHARGEMENT] {label} : téléchargement / chargement de {mid}…", flush=True)
            try:
                model, tok, mode = load_model(mid)
                print(f"[CHARGEMENT] {label} : {mid} en {mode}")
                break
            except Exception as e:
                print(f"[CHARGEMENT] échec {mid} : {type(e).__name__}: {str(e)[:300]}")
                model = None
        if model is None:
            raise RuntimeError("modèle non chargé")
        if getattr(tok, "pad_token_id", None) is None:
            tok.pad_token = tok.eos_token
        enc = [tok.encode(p)[:12] if SMOKE_TEST else tok.encode(p) for p in PROMPTS]     # comme A1k et A1p
        g = Gen(model, tok, label)
        res, info = run_main(g, enc)
        if label in A1P_MD5 and not SMOKE_TEST:
            m5 = {"temoin": md5(res["temoin"]["tok"]), "choc1": md5(res["choc1"]["tok"]), "X": md5(info["X"]), "Y": md5(info["Y"])}
            okm = {k: m5[k] == A1P_MD5[label][k] for k in m5}
            print(f"[ANCRAGE] {label} : jetons identiques à A1p (MD5, 768 × 128) : "
                  + ", ".join(f"{k} {v}" for k, v in okm.items())
                  + ("" if all(okm.values()) else " (écart numérique possible ; les contrastes, mesurés ici, restent valides)"))
        valide = (res["temoin"]["tok"][:, T2] != info["X"]) & (res["choc1"]["tok"][:, T2] != info["X"])
        P = int(valide.sum())

        abl = Ablation(model)
        nl, nh = abl.nl, abl.nh
        ids_sc = suites_repetees(g, SEED + 11)
        sc_att = score_induction(g, ids_sc)
        verif_crochet(g, abl, res, enc, choisir(sc_att, 1)[0])
        t_sc = time.time()
        sc = score_recopie(g, abl, ids_sc)
        K = max(1, round(K_FRAC * nl * nh))
        sel, motif = choisir(sc, K), choisir(sc_att, K)
        rngA = np.random.default_rng(SEED + 13)
        aleas = [tirage_apparie(sel, nl, nh, rngA) for _ in range(N_ALEA)]
        conds = {"aucune": [], "induction": sel, "motif": motif}
        for d in K_DOSES:
            conds[f"dose_{d:.0%}"] = choisir(sc, max(1, round(d * nl * nh)))
        conds.update({f"alea_{j}": a for j, a in enumerate(aleas)})
        sc_al = [float(np.mean([sc[t] for t in a])) for a in aleas]
        print(f"[INDUCTION] {label} : {nl} couches × {nh} têtes ; score de recopie calculé en {time.time() - t_sc:.0f} s : "
              f"médiane {np.median(sc):+.3f} nats, 99e centile {np.percentile(sc, 99):+.3f} ; K = {K} têtes retenues : "
              + ", ".join(f"{nom_tete(t)} {sc[t]:+.2f}" for t in sel))
        print(f"[INDUCTION] {label} : score d'attention (Olsson) médian {np.median(sc_att):.3f}, 99e centile "
              f"{np.percentile(sc_att, 99):.3f} ; ensemble « motif » : " + ", ".join(f"{nom_tete(t)} {sc_att[t]:.2f}" for t in motif)
              + f" ; recouvrement avec l'ensemble retenu : {len(set(sel) & set(motif))} sur {K}")
        print(f"[INDUCTION] {label} : score de recopie moyen des têtes retenues {np.mean([sc[t] for t in sel]):+.3f} ; des "
              f"tirages aléatoires appariés par couche : médiane {np.median(sc_al):+.3f} [{min(sc_al):+.3f} ; {max(sc_al):+.3f}]")

        ids_m1 = suites_repetees(g, SEED + 17)                 # suites neuves, distinctes de celles du score
        perte = {}
        for c, heads in conds.items():
            abl.set(heads)
            perte[c] = perte_copie(g, ids_m1)
        abl.clear()
        d1 = {c: perte[c][1] - perte["aucune"][1] for c in conds}
        d_al = [d1[c] for c in conds if c.startswith("alea_")]
        m1 = regle_m1(d1["induction"], d_al)
        print(f"[M1] {label} : perte sur la seconde copie sans ablation {perte['aucune'][1]:.2f} nats (première copie "
              f"{perte['aucune'][0]:.2f}) ; hausse sous ablation induction {d1['induction']:+.2f} ; aléatoire médiane "
              f"{np.median(d_al):+.2f} [{min(d_al):+.2f} ; {max(d_al):+.2f}] ; première copie sous ablation induction "
              f"{perte['induction'][0] - perte['aucune'][0]:+.2f} → {m1}")
        for d in K_DOSES:
            print(f"[M1] {label} : dose {d:.0%} : hausse {d1[f'dose_{d:.0%}']:+.2f}")
        print(f"[M1] {label} : ensemble « motif » (score d'attention) : hausse {d1['motif']:+.2f}")

        SRs, E = {}, {}
        t_rep = time.time()
        for ci_, (c, heads) in enumerate(conds.items()):
            abl.set(heads)
            SRs[c] = run_replay(g, enc, res, info)
            abl.clear()
            E[c] = {k: v[valide] for k, v in effets_cond(SRs[c]).items()}
            if ci_ == 0:
                dt = time.time() - t_rep
                print(f"[PROGRESSION] {label} : une condition rejouée en {dt:.0f} s ; estimation ~{dt * len(conds) / 60:.0f} min "
                      f"pour {len(conds)} conditions", flush=True)
                gen = res["choc1"]["S"][:, T2:T2 + NPOS]
                dd = np.abs(SRs[c]["choc1"] - gen)
                dp = max(float(np.abs(SRs[c][a][:, 0] - SRs[c]["choc1"][:, 0]).max()) for a in BRAS_R)
                print(f"[VÉRIF] {label} choc1 rejoué contre généré : écart moyen {dd.mean():.1e}, max {dd.max():.1e} ; "
                      f"bras identiques avant t₂ (indice t₂) : écart max {dp:.1e}", flush=True)
                if dd.mean() > 0.1 or dp > 0.05:
                    raise RuntimeError("passe rejouée incohérente : résultats non interprétables")
                if dt * len(conds) / 3600 > MAX_HOURS:
                    raise TimeoutError(f"estimation {dt * len(conds) / 60:.0f} min > budget {MAX_HOURS} h")
        print(f"[MESURE] {label} passes rejouées : {time.time() - t_rep:.0f} s", flush=True)

        rngB = np.random.default_rng(SEED + 3)
        boots = [rngB.integers(0, P, P) for _ in range(BOOT_N)]
        J = juger(E, boots, m1, ok)
        a = {"exclus": float(1 - valide.mean()), "n": P, "K": K, "tetes_induction": [nom_tete(t) for t in sel],
             "score_recopie": sc.tolist(), "score_attention": sc_att.tolist(),
             "tetes_motif": [nom_tete(t) for t in motif], "aleas": [[nom_tete(t) for t in al] for al in aleas],
             "perte_copie": perte, "M1": m1, **J}
        r0 = A1P_REF.get(label, {})
        (sp0, c0), (spi, cI) = J["Sp_aucune"], J["Sp_induction"]
        (r10, c10), (r1i, c1i) = J["R1_aucune"], J["R1_induction"]
        print(f"\n{'=' * 100}\n[RÉSULTATS] {label} (T = {T_MAIN[0]}) : {P} prompts retenus, {a['exclus']:.1%} exclus")
        print(f"[EFFETS] sans ablation : Sp = {sp0:.3f} [{c0[0]:.3f} ; {c0[1]:.3f}] (A1p {r0.get('Sp', float('nan')):.3f}) ; "
              f"R1 = {r10:.2f} [{c10[0]:.2f} ; {c10[1]:.2f}] (A1p {r0.get('R1', float('nan')):.2f})")
        print(f"[EFFETS] ablation induction : Sp = {spi:.3f} [{cI[0]:.3f} ; {cI[1]:.3f}] ; R1 = {r1i:.2f} [{c1i[0]:.2f} ; {c1i[1]:.2f}] ; "
              f"D_nouv = {E['induction']['nouv'].mean():+.4f} IC95 [{J['D_nouv_induction_ci'][0]:+.4f} ; {J['D_nouv_induction_ci'][1]:+.4f}]")
        for key, nom in (("dSp", "ΔSp"), ("dR1", "ΔR1")):
            d, dc = J[key]
            dal = J[key + "_alea"]
            rang = 1 + sum(x >= d for x in dal)
            print(f"[EFFETS] {nom} induction = {d:+.3f} IC95 [{dc[0]:+.3f} ; {dc[1]:+.3f}] ; aléatoire (témoin − référence) : "
                  f"médiane {np.median(dal):+.3f} [{min(dal):+.3f} ; {max(dal):+.3f}] ; rang de l'induction {rang} sur {len(dal) + 1}")
        for d in list(K_DOSES) + ["motif"]:
            e = E["motif" if d == "motif" else f"dose_{d:.0%}"]
            print(f"[DOSE] {label} {'motif (score d’attention)' if d == 'motif' else f'{d:.0%} des têtes'} : Sp = {e['rep'].mean() / e['nouv'].mean():.3f} ; "
                  f"R1 = {e['rep1'].mean() / e['nouv1'].mean():.2f}")
        prof = {c: tranches(SRs[c], valide) for c in conds}
        prof_al = np.array([prof[c] for c in conds if c.startswith("alea_")])
        for c in ("aucune", "induction"):
            print(f"[PROFIL] {label} {c:<10} D_rep/D_nouv par tranches de k (1 | 2–7 | 8–15 | 16–31) : "
                  + " ".join(f"{x:.2f}" for x in prof[c]))
        print(f"[PROFIL] {label} aléatoire  (médiane des tirages)                         : "
              + " ".join(f"{x:.2f}" for x in np.median(prof_al, 0)))
        s0 = {c: float(E[c]["S0"].mean()) for c in conds}
        s0_al = [s0[c] for c in conds if c.startswith("alea_")]
        print(f"[DÉGÂT] {label} entropie de base de choc1 (k = 2–31) : aucune {s0['aucune']:.3f} ; induction {s0['induction']:.3f} ; "
              f"aléatoire médiane {np.median(s0_al):.3f} [{min(s0_al):.3f} ; {max(s0_al):.3f}]")
        print(f"[V1] {label} recopie à k = 1 : {J['V1']}")
        print(f"[V2] {label} habituation sur k = 2–31 : {J['V2']}")
        a.update({"profil": prof, "S0": s0, "minutes": (time.time() - t_start) / 60})
        ALL[label] = a
        np.savez_compressed(os.path.join(OUT, f"{label}_a1q.npz"), valide=valide, score_recopie=sc, score_attention=sc_att,
                            tok_temoin=res["temoin"]["tok"], tok_choc1=res["choc1"]["tok"],
                            **{f"info_{k}": v for k, v in info.items()},
                            **{f"S_{c}_{b}": SRs[c][b] for c in SRs for b in BRAS_R})
        print(f"[DURÉE] {label} : {a['minutes']:.1f} min", flush=True)
    except Exception as e:
        print(f"[ERREUR] {label} : {type(e).__name__}: {str(e)[:500]}", flush=True)
        ALL[label] = {"erreur": f"{type(e).__name__}: {str(e)[:300]}"}
    if abl is not None:
        abl.remove()
    del model, tok, g, abl
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    ecrire_resume()

In [ ]:
# ---------- Synthèse ----------
print(f"\n{'=' * 100}\n[SYNTHÈSE] A1q — l'habituation d'A1p passe-t-elle par les têtes d'induction ?")
v2 = {m: ALL[m]["V2"] for m in ALL if "V2" in ALL[m]}
for m, v in v2.items():
    print(f"[SYNTHÈSE] {m:<22} : M1 {ALL[m]['M1']:<10} ; V1 {ALL[m]['V1']:<24} ; V2 {v}")
these = "NON TESTABLE (modèle manquant)" if len(v2) < len(MODELS) else synthese(v2, ok)
ALL["these"] = these
print(f"[SYNTHÈSE] thèse « l'habituation d'A1p passe par les têtes d'induction » : {these} "
      f"(prédiction SOUTENUE 15 %, PARTIELLE 30 %, RÉFUTÉE 20 %, DÉPEND DU MODÈLE 15 %)")
ecrire_resume()
print(f"[SORTIE] fichiers dans {OUT} : {sorted(os.listdir(OUT))}")